# Prediksi TC dengan model pohon dan SHAP

Targetnya `tc_resmi_pct` per kabupaten/kota. Latih model dengan fitur dan label 2023, pilih konfigurasi memakai MAE validasi 2024, lalu uji model yang dikunci di 2025. Prediksi berasal langsung dari satu model terlatih. TC resmi historis tidak dimasukkan sebagai fitur atau dicampur ke prediksi.

Jalankan dari folder proyek dengan Python yang memiliki `pandas`, `numpy`, `openpyxl`, `scikit-learn`, `xgboost`, `lightgbm`, `catboost`, dan `plotly`.

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from catboost import CatBoostRegressor, Pool
from IPython.display import display
from lightgbm import LGBMRegressor
from plotly.subplots import make_subplots
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import DMatrix, XGBRegressor

In [2]:
DATA_DIR = Path("data")
TARGET = "tc_resmi_pct"
INCIDENCE = "estimasi_insiden_tbc"
MAX_NEIGHBORS = 5
YEARS = (2023, 2024, 2025)
LOCAL_FEATURES = [
    "penduduk_jiwa", "kepadatan_jiwa_km2", "penduduk_miskin_pct",
    "rt_hunian_layak_pct", "puskesmas_per_100rb_penduduk", "wilayah_kota",
]


def workbook(year):
    return DATA_DIR / f"TRACK-TB_Join_Sumatera_{year}.xlsx"


def load_year(year):
    frame = pd.read_excel(workbook(year), sheet_name=f"JOIN_{year}", dtype={"kode_bps": str})
    frame = frame.set_index("kode_bps").sort_index()
    if not frame.index.is_unique or frame.index.hasnans or not frame["tahun"].eq(year).all():
        raise ValueError(f"Kode BPS atau tahun tidak valid pada JOIN_{year}.")
    frame["wilayah_kota"] = frame["jenis_wilayah"].eq("Kota").astype(int)
    if not np.isfinite(frame[LOCAL_FEATURES]).to_numpy().all():
        raise ValueError(f"Ada fitur wilayah yang kosong atau tidak valid pada {year}.")
    return frame


frames = {year: load_year(year) for year in YEARS}
codes = frames[2023].index
if any(not frame.index.equals(codes) for frame in frames.values()):
    raise ValueError("Daftar kode BPS berbeda antartahun.")

coverage = pd.DataFrame({
    "TC resmi": {year: frame[TARGET].count() for year, frame in frames.items()},
    "TC kosong": {year: frame[TARGET].isna().sum() for year, frame in frames.items()},
    "estimasi insiden asli": {year: frame[INCIDENCE].count() for year, frame in frames.items()},
})
print(coverage.to_string())

      TC resmi  TC kosong  estimasi insiden asli
2023       154          0                    154
2024        71         83                     26
2025        57         97                      0


In [3]:
def load_edges():
    nodes = pd.read_excel(
        workbook(2023), sheet_name="GNN_NODES", dtype={"kode_bps": str},
        usecols=["node_id", "kode_bps"],
    )
    if not nodes["node_id"].is_unique or not pd.Index(nodes["kode_bps"]).sort_values().equals(codes):
        raise ValueError("GNN_NODES tidak cocok dengan JOIN_2023.")
    id_to_code = nodes.set_index("node_id")["kode_bps"]
    edges = pd.read_excel(
        workbook(2023), sheet_name="GNN_EDGES",
        dtype={"kode_bps_a": str, "kode_bps_b": str},
        usecols=["src_node_id", "dst_node_id", "kode_bps_a", "kode_bps_b", "panjang_batas_aproksimasi_km"],
    ).rename(columns={"panjang_batas_aproksimasi_km": "batas_km"})
    for node_col, code_col in (("src_node_id", "kode_bps_a"), ("dst_node_id", "kode_bps_b")):
        if not edges[node_col].map(id_to_code).eq(edges[code_col]).all():
            raise ValueError("Kode BPS pada GNN_EDGES tidak cocok dengan ID simpul.")
    if edges["kode_bps_a"].eq(edges["kode_bps_b"]).any() or not edges["batas_km"].gt(0).all():
        raise ValueError("Ada edge ke diri sendiri atau panjang batas tidak valid.")
    return edges[["kode_bps_a", "kode_bps_b", "batas_km"]]


edges = load_edges()
directed = pd.concat([
    edges.rename(columns={"kode_bps_a": "wilayah", "kode_bps_b": "tetangga"}),
    edges.rename(columns={"kode_bps_b": "wilayah", "kode_bps_a": "tetangga"}),
], ignore_index=True)
if directed.duplicated(["wilayah", "tetangga"]).any():
    raise ValueError("Ada pasangan tetangga ganda.")
ranked = directed.sort_values(
    ["wilayah", "batas_km", "tetangga"], ascending=[True, False, True],
)
selected_edges = ranked.groupby("wilayah", sort=False).head(MAX_NEIGHBORS)
total_neighbors = directed.groupby("wilayah").size().reindex(codes, fill_value=0)
used_neighbors = selected_edges.groupby("wilayah").size().reindex(codes, fill_value=0)
selected_neighbor_codes = (
    selected_edges.groupby("wilayah")["tetangga"].agg("|".join).reindex(codes, fill_value="")
)

assert np.array_equal(used_neighbors, total_neighbors.clip(upper=MAX_NEIGHBORS))

incidence_2023 = frames[2023][INCIDENCE]
if not np.isfinite(incidence_2023).all() or incidence_2023.lt(0).any():
    raise ValueError("Estimasi insiden 2023 harus terisi dan bernilai nonnegatif.")

print(f"Graf awal: {len(codes)} wilayah, {len(edges)} pasangan batas.")
print(f"Wilayah dengan tetangga dipotong ke lima: {total_neighbors.gt(MAX_NEIGHBORS).sum()}.")
print("Distribusi jumlah tetangga yang digunakan")
print(used_neighbors.value_counts().sort_index().to_string())

Graf awal: 154 wilayah, 315 pasangan batas.
Wilayah dengan tetangga dipotong ke lima: 43.
Distribusi jumlah tetangga yang digunakan
0     8
1    20
2    14
3    18
4    26
5    68


### Fitur per wilayah dan per tetangga

Model memakai kepadatan, jumlah penduduk, kemiskinan, hunian layak, puskesmas per 100 ribu penduduk, dan klasifikasi kota/kabupaten di wilayah target. Untuk fitur insiden, satu kolom mewakili satu tetangga langsung. Peringkat mengikuti panjang batas administratif, dengan kode BPS sebagai pemecah seri. Maksimal lima kolom tetangga diisi. Model pohon menangani kolom yang kosong ketika tetangga lebih sedikit.

Ada tiga profil fitur. Profil lokal menjadi pembanding, profil insiden menambahkan estimasi jumlah kasus setiap tetangga, dan profil laju menambahkan estimasi kasus tiap tetangga per 100 ribu penduduk. Kedua profil spasial mengambil estimasi dan penyebut dari data 2023 karena estimasi 2025 kosong. Fitur demografi dan layanan wilayah target mengikuti tahun yang diprediksi. Estimasi insiden wilayah target, kasus dilaporkan, selisih notifikasi, dan TC historis dikeluarkan dari semua kolom input.

In [4]:
slots = selected_edges.assign(
    peringkat=selected_edges.groupby("wilayah", sort=False).cumcount().add(1),
).pivot(index="wilayah", columns="peringkat", values="tetangga")
slots = slots.reindex(index=codes, columns=range(1, MAX_NEIGHBORS + 1))
assert slots.notna().sum(axis=1).eq(used_neighbors).all()
assert not slots.eq(pd.Series(codes, index=codes), axis=0).any().any()

neighbor_population = frames[2023]["penduduk_jiwa"]
if not np.isfinite(neighbor_population).all() or not neighbor_population.gt(0).all():
    raise ValueError("Penduduk 2023 harus terisi dan lebih besar dari nol.")
incidence_rate_2023 = 100_000 * incidence_2023 / neighbor_population


def neighbor_columns(values, prefix):
    return pd.DataFrame({
        f"{prefix}_{rank}": slots[rank].map(values).astype(float)
        for rank in slots.columns
    }, index=codes)


spatial_features = {
    "insiden tetangga": neighbor_columns(incidence_2023, "kasus_tetangga"),
    "laju tetangga": neighbor_columns(incidence_rate_2023, "laju_tetangga"),
}
features = {"lokal": {year: frame[LOCAL_FEATURES] for year, frame in frames.items()}}
features.update({
    name: {year: local.join(spatial) for year, local in features["lokal"].items()}
    for name, spatial in spatial_features.items()
})
print("Jumlah kolom input per profil")
print(pd.Series({name: table[2023].shape[1] for name, table in features.items()}).to_string())

Jumlah kolom input per profil
lokal                6
insiden tetangga    11
laju tetangga       11


### Pelatihan dan pemilihan model

XGBoost, LightGBM, dan CatBoost membangun pohon secara bertahap. Untuk tiap keluarga, uji konfigurasi pohon dangkal dan fungsi rugi kuadrat atau Huber. MAE 2024 menentukan pemenang sebelum label 2025 dipakai. Hanya profil dengan estimasi tetangga yang menjadi kandidat untuk mengisi data 2025. Profil lokal menunjukkan hasil saat hanya memakai informasi wilayah sendiri.

In [5]:
catalog = {
    "XGBoost": (
        XGBRegressor,
        dict(tree_method="hist", n_jobs=1, random_state=42, base_score=0.5,
             min_child_weight=5, learning_rate=0.05, reg_lambda=2),
        {
            "L2 d1": dict(objective="reg:squarederror", max_depth=1, n_estimators=160),
            "L2 d2": dict(objective="reg:squarederror", max_depth=2, n_estimators=140),
            "L2 d3": dict(objective="reg:squarederror", max_depth=3, n_estimators=110),
            "pseudo-Huber d2": dict(objective="reg:pseudohubererror", huber_slope=0.18,
                                    max_depth=2, n_estimators=160, min_child_weight=2),
        },
    ),
    "LightGBM": (
        LGBMRegressor,
        dict(n_jobs=1, random_state=42, verbosity=-1, deterministic=True, force_col_wise=True,
             min_child_samples=10, learning_rate=0.05, reg_lambda=1, boost_from_average=False),
        {
            "L2 d1": dict(objective="regression", max_depth=1, num_leaves=2, n_estimators=160),
            "L2 d2": dict(objective="regression", max_depth=2, num_leaves=4, n_estimators=140),
            "L2 d3": dict(objective="regression", max_depth=3, num_leaves=8, n_estimators=110),
            "Huber d2": dict(objective="huber", alpha=0.18, max_depth=2,
                             num_leaves=4, n_estimators=200),
        },
    ),
    "CatBoost": (
        CatBoostRegressor,
        dict(thread_count=1, random_seed=42, verbose=False, allow_writing_files=False,
             learning_rate=0.05, l2_leaf_reg=4, boost_from_average=False),
        {
            "L2 d2": dict(loss_function="RMSE", depth=2, iterations=160),
            "L2 d3": dict(loss_function="RMSE", depth=3, iterations=140),
            "L2 d4": dict(loss_function="RMSE", depth=4, iterations=110),
            "Huber d3": dict(loss_function="Huber:delta=0.18", depth=3, iterations=180),
        },
    ),
}
target = frames[2023][TARGET]
if not np.isfinite(target).all():
    raise ValueError("Label pelatihan 2023 harus lengkap.")

fitted = {}
for family, (constructor, common, variants) in catalog.items():
    for setting, options in variants.items():
        for profile, yearly in features.items():
            name = f"{family} {setting} [{profile}]"
            model = constructor(**(common | options))
            model.fit(yearly[2023], target / 100)
            fitted[name] = (family, profile, model)


def predict(name, year):
    _, profile, model = fitted[name]
    values = np.asarray(model.predict(features[profile][year]), dtype=float)
    return pd.Series(100 * values, index=codes, name=name)


def metrics(actual, prediction):
    predicted = prediction.loc[actual.index]
    return {
        "n": len(actual),
        "MAE_pp": mean_absolute_error(actual, predicted),
        "RMSE_pp": np.sqrt(mean_squared_error(actual, predicted)),
        "R2": r2_score(actual, predicted),
    }


actual_2024 = frames[2024][TARGET].dropna()
validation = pd.DataFrame.from_dict({
    name: {"keluarga": family, "profil": profile, **metrics(actual_2024, predict(name, 2024))}
    for name, (family, profile, _) in fitted.items()
}, orient="index").sort_values("MAE_pp", kind="stable")

with_neighbors = validation[validation["profil"].ne("lokal")]
selected = with_neighbors.index[0]
best_by_family = {
    family: with_neighbors[with_neighbors["keluarga"].eq(family)].index[0] for family in catalog
}
best_local = validation[validation["profil"].eq("lokal")].index[0]
print(f"{len(fitted)} konfigurasi dilatih pada {len(target)} wilayah tahun 2023.")
print("\n10 konfigurasi spasial terbaik pada validasi 2024")
print(with_neighbors[["n", "MAE_pp", "RMSE_pp", "R2"]].head(10).round(3).to_string())
print(f"\nModel terpilih: {selected}")

36 konfigurasi dilatih pada 154 wilayah tahun 2023.

10 konfigurasi spasial terbaik pada validasi 2024
                                             n  MAE_pp  RMSE_pp     R2
XGBoost L2 d3 [insiden tetangga]            71  13.877   18.274  0.433
LightGBM Huber d2 [insiden tetangga]        71  14.179   18.938  0.391
XGBoost pseudo-Huber d2 [insiden tetangga]  71  14.333   18.856  0.397
XGBoost L2 d3 [laju tetangga]               71  14.633   18.871  0.396
XGBoost L2 d1 [laju tetangga]               71  14.666   20.152  0.311
XGBoost L2 d2 [laju tetangga]               71  14.765   19.688  0.342
LightGBM Huber d2 [laju tetangga]           71  14.794   19.681  0.343
XGBoost pseudo-Huber d2 [laju tetangga]     71  14.958   19.662  0.344
CatBoost Huber d3 [insiden tetangga]        71  15.000   19.360  0.364
XGBoost L2 d2 [insiden tetangga]            71  15.009   19.551  0.351

Model terpilih: XGBoost L2 d3 [insiden tetangga]


### Test 2025

Pemenang tiap keluarga, pemenang lokal, dan model yang mengisi TC 2025 dipilih dari validasi 2024. Bobot model tetap berasal dari 2023. Test memakai 57 wilayah berlabel di empat provinsi. Data 2025 pernah digunakan untuk evaluasi sebelumnya, sehingga angka di bawah merupakan evaluasi retrospektif untuk percobaan ini.

In [6]:
actual_2025 = frames[2025][TARGET].dropna()
finalists = list(dict.fromkeys([selected, *best_by_family.values(), best_local]))
test_predictions = {name: predict(name, 2025) for name in finalists}
test = pd.DataFrame.from_dict({
    name: metrics(actual_2025, prediction) for name, prediction in test_predictions.items()
}, orient="index")
test.insert(1, "MAE_validasi_2024_pp", validation.loc[test.index, "MAE_pp"])
test = test.rename(columns={"MAE_pp": "MAE_test_2025_pp"})
print(test.round(3).to_string())

missing_2025 = frames[2025][TARGET].isna()
selected_prediction = test_predictions[selected]
if not np.isfinite(selected_prediction).all() or selected_prediction.lt(0).any():
    raise ValueError("Prediksi harus terisi dan nonnegatif.")

                                       n  MAE_validasi_2024_pp  MAE_test_2025_pp  RMSE_pp     R2
XGBoost L2 d3 [insiden tetangga]      57                13.877            13.688   19.433  0.243
LightGBM Huber d2 [insiden tetangga]  57                14.179            14.755   20.676  0.143
CatBoost Huber d3 [insiden tetangga]  57                15.000            14.821   19.842  0.210
XGBoost L2 d1 [lokal]                 57                15.372            14.605   20.751  0.136


### Penjelasan SHAP

Nilai SHAP dihitung langsung dari model pohon. Nilai dasar ditambah kontribusi semua fitur harus sama dengan prediksi TC sebelum pembulatan. Kontribusi positif menaikkan prediksi dan kontribusi negatif menurunkannya, dalam poin persentase. Ringkasan global menjumlahkan magnitudo kontribusi di 97 wilayah yang TC resminya kosong. Penjelasan lokal juga menampilkan kode BPS yang benar-benar menempati tiap peringkat tetangga.

[TreeSHAP XGBoost](https://xgboost.readthedocs.io/en/stable/prediction.html) · [Kontribusi LightGBM](https://lightgbm.readthedocs.io/en/stable/Parameters.html#predict_contrib) · [Nilai SHAP CatBoost](https://catboost.ai/docs/en/concepts/shap-values?lang=en)

In [7]:
def shap_values(name):
    family, profile, model = fitted[name]
    matrix = features[profile][2025]
    if family == "XGBoost":
        contributions = model.get_booster().predict(DMatrix(matrix), pred_contribs=True)
    elif family == "LightGBM":
        contributions = model.booster_.predict(matrix, pred_contrib=True, num_threads=1)
    else:
        contributions = model.get_feature_importance(Pool(matrix), type="ShapValues", thread_count=1)
    values = np.asarray(contributions, dtype=float) * 100
    impact = pd.DataFrame(values[:, :-1], index=codes, columns=matrix.columns)
    base = pd.Series(values[:, -1], index=codes, name="nilai_dasar_pct")
    np.testing.assert_allclose(
        (base + impact.sum(axis=1)).to_numpy(), test_predictions[name].to_numpy(),
        rtol=0, atol=5e-4,
    )
    return base, impact


explanations = {name: shap_values(name) for name in best_by_family.values()}
base, impact = explanations[selected]
selected_profile = fitted[selected][1]
selected_input = features[selected_profile][2025]
importance = impact.loc[missing_2025].abs().sum().sort_values(ascending=False)
print("Kontribusi SHAP absolut kumulatif pada wilayah yang diprediksi (poin persentase)")
print(importance.round(2).to_string())

example_code = codes[missing_2025 & used_neighbors.gt(0)][0]
example_name = frames[2025].at[example_code, "kabupaten_kota"]
neighbor_code_by_feature = dict(zip(spatial_features[selected_profile].columns, slots.loc[example_code]))
local_explanation = pd.DataFrame({
    "nilai_fitur": selected_input.loc[example_code],
    "kontribusi_pp": impact.loc[example_code],
})
local_explanation["kode_BPS_tetangga"] = pd.Series(neighbor_code_by_feature)
local_order = local_explanation["kontribusi_pp"].abs().sort_values(ascending=False).index
local_explanation = local_explanation.loc[local_order]
print(f"\nContoh {example_name} ({example_code})")
print(f"Nilai dasar {base.loc[example_code]:.2f}%")
print(f"Kontribusi bersih {impact.loc[example_code].sum():+.2f} poin persentase")
print(f"Prediksi model {selected_prediction.loc[example_code]:.2f}%\n")
print(local_explanation.round(3).to_string(na_rep=""))
print(f"\nPenjumlahan SHAP lolos pada {len(codes)} wilayah untuk tiap keluarga model.")

Kontribusi SHAP absolut kumulatif pada wilayah yang diprediksi (poin persentase)
kepadatan_jiwa_km2              605.81
kasus_tetangga_3                373.93
kasus_tetangga_4                248.70
penduduk_jiwa                   221.99
kasus_tetangga_1                217.25
rt_hunian_layak_pct             205.79
kasus_tetangga_2                193.25
puskesmas_per_100rb_penduduk    148.09
wilayah_kota                    106.31
penduduk_miskin_pct              50.43
kasus_tetangga_5                 40.77

Contoh Aceh Singkil (1102)
Nilai dasar 56.30%
Kontribusi bersih -11.02 poin persentase
Prediksi model 45.28%

                              nilai_fitur  kontribusi_pp kode_BPS_tetangga
kasus_tetangga_1                  301.000         -5.878              1175
kasus_tetangga_3                  875.000          4.553              1103
kepadatan_jiwa_km2                 75.000         -3.920                  
kasus_tetangga_2                  180.000         -2.876              1216
kasu

In [8]:
LABELS = {
    "penduduk_jiwa": "Jumlah penduduk",
    "kepadatan_jiwa_km2": "Kepadatan penduduk",
    "penduduk_miskin_pct": "Penduduk miskin (%)",
    "rt_hunian_layak_pct": "Hunian layak (%)",
    "puskesmas_per_100rb_penduduk": "Puskesmas / 100 ribu",
    "wilayah_kota": "Status kota",
}
for position in range(1, MAX_NEIGHBORS + 1):
    LABELS[f"kasus_tetangga_{position}"] = f"Estimasi kasus tetangga {position}"
    LABELS[f"laju_tetangga_{position}"] = f"Laju insiden tetangga {position}"

global_bars = importance.head(7).iloc[::-1]
local_bars = local_explanation["kontribusi_pp"].head(7).iloc[::-1]
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Ringkasan wilayah dengan TC kosong", f"Prediksi {example_name}"),
    horizontal_spacing=0.34,
)
fig.add_trace(go.Bar(
    x=global_bars.to_numpy(), y=[LABELS[key] for key in global_bars.index],
    orientation="h", marker_color="#226d72", hovertemplate="%{y}<br>%{x:.2f} pp<extra></extra>",
), row=1, col=1)
fig.add_trace(go.Bar(
    x=local_bars.to_numpy(), y=[LABELS[key] for key in local_bars.index],
    orientation="h", marker_color=["#226d72" if value >= 0 else "#b45441" for value in local_bars],
    hovertemplate="%{y}<br>%{x:+.2f} pp<extra></extra>",
), row=1, col=2)
fig.update_xaxes(title_text="Total |SHAP| (pp)", row=1, col=1)
fig.update_xaxes(title_text="Kontribusi pada prediksi (pp)", zeroline=True, row=1, col=2)
fig.update_yaxes(automargin=True)
fig.update_layout(height=420, showlegend=False, template="plotly_white")
display({
    "application/vnd.plotly.v1+json": json.loads(fig.to_json()),
    "text/plain": "Grafik SHAP global dan contoh lokal. Nilai lengkap tersedia di tabel.",
}, raw=True)

Grafik SHAP global dan contoh lokal. Nilai lengkap tersedia di tabel.

### Ekspor prediksi 2025

`tc_percentage_prediksi` berisi keluaran langsung model terpilih. `tc_percentage` mempertahankan angka resmi saat tersedia dan memakai hasil model untuk wilayah dengan TC kosong. CSV juga memuat prediksi terbaik tiap keluarga model serta urutan kode BPS tetangga. Enam provinsi dengan TC kosong berada di luar cakupan pengukuran galat 2025.

In [9]:
results = frames[2025][["tahun", "provinsi", "kabupaten_kota", TARGET]].copy()
results["tc_percentage_prediksi"] = selected_prediction.round(2)
results["tc_percentage"] = results[TARGET].fillna(results["tc_percentage_prediksi"])
results["sumber_tc_percentage"] = np.where(missing_2025, "estimasi model", "resmi")
results["model_tc_percentage_prediksi"] = selected
for family, name in best_by_family.items():
    results[f"tc_percentage_prediksi_{family.lower()}"] = test_predictions[name].round(2)
results["jumlah_tetangga_tersedia"] = total_neighbors
results["jumlah_tetangga_dipakai"] = used_neighbors
results["kode_bps_tetangga"] = selected_neighbor_codes

assert results.index.is_unique and results["tc_percentage"].notna().all()
assert results.loc[~missing_2025, "tc_percentage"].eq(actual_2025).all()
assert results.loc[missing_2025, "tc_percentage"].eq(
    results.loc[missing_2025, "tc_percentage_prediksi"]
).all()

output = Path("prediksi_tc_2025.csv")
results.to_csv(output, encoding="utf-8-sig")
print(f"{len(results)} wilayah disimpan ke {output}")
print(results["sumber_tc_percentage"].value_counts().to_string())

154 wilayah disimpan ke prediksi_tc_2025.csv
sumber_tc_percentage
estimasi model    97
resmi             57


## 3. Simulasi alokasi skrining

Pertanyaannya adalah berapa kasus tambahan yang diperkirakan bisa ditemukan dengan jumlah skrining yang sama. Simulasi memakai 100.000 skrining per strategi dengan biaya per orang yang seragam. Anggaran dan faktor temu bisa diubah di sel berikut.

| Strategi | Aturan alokasi (informasi tahun t) |
|---|---|
| Merata | Bagi jumlah skrining sama rata per kabupaten/kota. |
| Berdasarkan penduduk | Bagi secara proporsional terhadap jumlah penduduk tahun evaluasi. |
| TRACK-TB | Bagi secara proporsional terhadap selisih TC positif tahun t, yaitu TC prediksi dikurangi TC resmi. |
| Beban kasus hilang | Bagi secara proporsional terhadap estimasi insiden tahun t × maks(0, 1 − TC resmi tahun t/100). |
| Kepadatan kasus hilang (tanpa model) | Beban kasus hilang dibagi jumlah penduduk tahun t. Pembanding tanpa model. |
| TC rendah (tanpa model) | Bagi secara proporsional terhadap 100 − TC resmi tahun t. Pembanding tanpa model. |

**Pengujian bergulir.** Alokasi selalu disusun dengan informasi tahun t, lalu dinilai dengan TC resmi tahun t + 1. Wilayah tanpa TC resmi pada kedua tahun tidak diikutsertakan.

| Pengujian | Informasi alokasi | Penilaian | Wilayah |
|---|---|---|---|
| 2024 | 2023 | TC resmi 2024 | 71 |
| 2025 | 2024 | TC resmi 2025 | 57 |

**Prediksi out-of-fold 2023.** Model terpilih dilatih dengan data 2023, sehingga prediksinya untuk 2023 tidak boleh dipakai langsung. TC prediksi 2023 dihitung dengan validasi silang lima lipatan, diulang dengan 10 pembagian acak, lalu prediksinya dirata-rata. Setiap wilayah diprediksi oleh model yang tidak melihat datanya.

**Selisih TC (poin persentase)** = TC prediksi − TC resmi. TRACK-TB memberi bobot pada bagian positif selisih.

**Estimasi insiden** tahun t memakai estimasi workbook bila tersedia. Bila kosong, laju insiden 2023 diterapkan pada jumlah penduduk tahun t. Estimasi insiden 2025 = insiden 2023 × penduduk 2025 / penduduk 2023.

**Estimasi kasus belum ternotifikasi** = estimasi insiden × maks(0, 1 − TC/100). [Definisi TC WHO](https://www.who.int/data/gho/indicator-metadata-registry/imr-details/1422)

**Ekspektasi kasus tambahan ditemukan** = estimasi kasus belum ternotifikasi × proporsi penduduk yang diskrining × faktor temu. Faktor temu 0,80 adalah asumsi ilustratif yang sama untuk semua strategi, sehingga peringkat strategi tidak bergantung pada nilainya.


In [10]:
from sklearn.base import clone
from sklearn.model_selection import KFold

ANGGARAN_SKRINING = 100_000
FAKTOR_TEMU = 0.80
JUMLAH_ULANGAN_OOF = 10

wilayah_skrining = frames[2025][["provinsi", "kabupaten_kota", "penduduk_jiwa"]].copy()
populasi_2023 = frames[2023]["penduduk_jiwa"]
populasi_2024 = frames[2024]["penduduk_jiwa"]
populasi_2025 = wilayah_skrining["penduduk_jiwa"]
if (
    not np.isfinite(populasi_2023).all()
    or not np.isfinite(populasi_2024).all()
    or not populasi_2024.gt(0).all()
    or not np.isfinite(populasi_2025).all()
    or not populasi_2023.gt(0).all()
    or not populasi_2025.gt(0).all()
    or not populasi_2025.eq(np.floor(populasi_2025)).all()
):
    raise ValueError("Populasi 2023–2025 harus terisi dan positif, dengan populasi 2025 dalam bilangan bulat.")

wilayah_skrining["penduduk_jiwa"] = populasi_2025.astype("int64")
wilayah_skrining["insiden_evaluasi"] = incidence_2023 * populasi_2025.div(populasi_2023)
wilayah_skrining["tc_prediksi_pct"] = selected_prediction


def estimasi_gap(insiden, tc):
    if not insiden.index.equals(tc.index) or not np.isfinite(tc).all():
        raise ValueError("TC harus tersedia untuk semua wilayah yang dihitung.")
    if not np.isfinite(insiden).all() or insiden.lt(0).any():
        raise ValueError("Estimasi insiden harus terisi dan nonnegatif.")
    return insiden * (1 - tc / 100).clip(lower=0, upper=1)


# Audit 2025: selisih pada tahun yang sama setelah TC resmi diketahui.
audit_tc_2025 = wilayah_skrining[["provinsi", "kabupaten_kota", "tc_prediksi_pct"]].copy()
audit_tc_2025["tc_asli_pct"] = frames[2025][TARGET]
audit_tc_2025["selisih_tc_pp"] = audit_tc_2025["tc_prediksi_pct"] - audit_tc_2025["tc_asli_pct"]

# Informasi 2024 untuk pengujian 2025.
prediksi_tc_2024 = predict(selected, 2024)
selisih_tc_2024 = prediksi_tc_2024 - frames[2024][TARGET]
estimasi_insiden_2024_sumber = frames[2024][INCIDENCE]
estimasi_insiden_2024 = estimasi_insiden_2024_sumber.fillna(incidence_2023 * populasi_2024.div(populasi_2023))
beban_2024 = estimasi_gap(estimasi_insiden_2024.loc[actual_2024.index], actual_2024)

# Informasi 2023 untuk pengujian 2024, dengan prediksi out-of-fold.
_, profil_terpilih, model_terpilih = fitted[selected]
X_2023 = features[profil_terpilih][2023]
y_2023 = frames[2023][TARGET]
prediksi_oof_ulangan = []
for seed in range(JUMLAH_ULANGAN_OOF):
    prediksi = pd.Series(np.nan, index=X_2023.index)
    for latih, uji in KFold(n_splits=5, shuffle=True, random_state=seed).split(X_2023):
        model_lipatan = clone(model_terpilih)
        model_lipatan.fit(X_2023.iloc[latih], y_2023.iloc[latih] / 100)
        prediksi.iloc[uji] = 100 * np.asarray(model_lipatan.predict(X_2023.iloc[uji]), dtype=float)
    prediksi_oof_ulangan.append(prediksi)
prediksi_tc_2023_oof = pd.concat(prediksi_oof_ulangan, axis=1).mean(axis=1)
selisih_tc_2023 = prediksi_tc_2023_oof - y_2023
beban_2023 = estimasi_gap(incidence_2023, y_2023)
print(f"MAE prediksi out-of-fold 2023: {(prediksi_tc_2023_oof - y_2023).abs().mean():.2f} pp")

pd.DataFrame({
    "provinsi": frames[2023]["provinsi"],
    "kabupaten_kota": frames[2023]["kabupaten_kota"],
    "tc_resmi_2023_pct": y_2023,
    "tc_prediksi_oof_2023_pct": prediksi_tc_2023_oof.round(4),
}).to_csv("prediksi_tc_2023_oof.csv", encoding="utf-8-sig")
print("Prediksi out-of-fold 2023 disimpan ke prediksi_tc_2023_oof.csv")


MAE prediksi out-of-fold 2023: 14.94 pp
Prediksi out-of-fold 2023 disimpan ke prediksi_tc_2023_oof.csv


In [11]:
def bagi_anggaran(kapasitas, bobot, anggaran):
    """Alokasi proporsional dengan batas penduduk dan pembulatan sisa terbesar."""
    if (
        isinstance(anggaran, bool)
        or not isinstance(anggaran, (int, np.integer))
        or not 0 < anggaran <= kapasitas.sum()
    ):
        raise ValueError("Anggaran harus bilangan bulat positif dan tidak melebihi total penduduk.")
    if not kapasitas.index.equals(bobot.index) or not np.isfinite(bobot).all() or bobot.lt(0).any():
        raise ValueError("Bobot harus terisi, sebaris dengan kapasitas, dan nonnegatif.")

    alokasi = pd.Series(0, index=kapasitas.index, dtype="int64")
    sisa_anggaran = anggaran
    while sisa_anggaran:
        ruang = kapasitas - alokasi
        bobot_aktif = bobot.where(ruang.gt(0), 0)
        if not bobot_aktif.gt(0).any():
            bobot_aktif = ruang

        kuota = sisa_anggaran * bobot_aktif / bobot_aktif.sum()
        penuh = ruang.gt(0) & kuota.ge(ruang)
        if penuh.any():
            alokasi.loc[penuh] += ruang.loc[penuh]
            sisa_anggaran -= int(ruang.loc[penuh].sum())
            continue

        bulat = np.floor(kuota).astype("int64")
        alokasi += bulat
        sisa_anggaran -= int(bulat.sum())
        if sisa_anggaran:
            pecahan = (kuota - bulat).loc[ruang.gt(bulat) & bobot_aktif.gt(0)]
            penerima = pecahan.sort_index().sort_values(ascending=False, kind="stable").head(sisa_anggaran).index
            alokasi.loc[penerima] += 1
        break

    assert alokasi.sum() == anggaran
    assert alokasi.ge(0).all() and alokasi.le(kapasitas).all()
    return alokasi


def simulasi_skrining(wilayah, tc_evaluasi, anggaran, faktor_temu, *, prioritas):
    """Kolom wilayah: penduduk_jiwa (kapasitas) dan insiden_evaluasi (insiden tahun penilaian)."""
    if not np.isfinite(faktor_temu) or not 0 <= faktor_temu <= 1:
        raise ValueError("Faktor temu harus berada di antara 0 dan 1.")
    populasi = wilayah["penduduk_jiwa"]
    bobot = {
        "Merata": pd.Series(1, index=wilayah.index),
        "Berdasarkan penduduk": populasi,
        **prioritas,
    }
    alokasi = pd.DataFrame({
        strategi: bagi_anggaran(populasi, nilai, anggaran) for strategi, nilai in bobot.items()
    })
    gap_evaluasi = estimasi_gap(wilayah["insiden_evaluasi"], tc_evaluasi)
    kasus_wilayah = alokasi.div(populasi, axis=0).mul(gap_evaluasi * faktor_temu, axis=0)
    hasil = pd.DataFrame({
        "skrining": alokasi.sum(),
        "ekspektasi_kasus": kasus_wilayah.sum(),
    })
    hasil["kasus_per_1000_skrining"] = 1000 * hasil["ekspektasi_kasus"] / hasil["skrining"]
    for acuan, nama in (("Merata", "merata"), ("Berdasarkan penduduk", "penduduk")):
        hasil[f"selisih_vs_{nama}"] = hasil["ekspektasi_kasus"] - hasil.at[acuan, "ekspektasi_kasus"]

    assert kasus_wilayah.ge(0).all().all()
    assert kasus_wilayah.le(gap_evaluasi, axis=0).all().all()
    return hasil, alokasi, kasus_wilayah

### Dua pengujian bergulir dan audit selisih TC

- **Pengujian 2024 (71 wilayah).** Bobot disusun dari informasi 2023, termasuk selisih TC 2023 dari prediksi out-of-fold. Hasil dinilai dengan TC resmi 2024.
- **Pengujian 2025 (57 wilayah).** Bobot disusun dari informasi 2024. Hasil dinilai dengan TC resmi 2025.
- **Ketahanan.** Pengujian 2024 juga dijalankan per pembagian out-of-fold untuk melihat rentang hasil TRACK-TB.
- **Audit TC 2025.** TC prediksi dikurangi TC resmi 2025 pada wilayah yang TC resminya tersedia.

Temuan adalah ekspektasi model, bukan hasil lapangan. Perbandingan jumlah kasus dilakukan di dalam masing-masing pengujian karena cakupan wilayahnya berbeda.


In [12]:
def bobot_strategi(selisih, beban, tc, populasi):
    return {
        "TRACK-TB": selisih.clip(lower=0),
        "Beban kasus hilang": beban,
        "Kepadatan kasus hilang (tanpa model)": beban / populasi,
        "TC rendah (tanpa model)": (100 - tc).clip(lower=0),
    }


def pengujian(tahun_info, tahun_uji, selisih, beban, insiden_uji):
    tc_info = frames[tahun_info][TARGET]
    tc_uji = frames[tahun_uji][TARGET]
    kode = tc_uji.dropna().index.intersection(tc_info.dropna().index).intersection(selisih.dropna().index)
    wilayah = pd.DataFrame({
        "penduduk_jiwa": frames[tahun_uji].loc[kode, "penduduk_jiwa"].round().astype("int64"),
        "insiden_evaluasi": insiden_uji.loc[kode],
    })
    prioritas = bobot_strategi(selisih.loc[kode], beban.loc[kode], tc_info.loc[kode], frames[tahun_info].loc[kode, "penduduk_jiwa"])
    return (wilayah, *simulasi_skrining(wilayah, tc_uji.loc[kode], ANGGARAN_SKRINING, FAKTOR_TEMU, prioritas=prioritas))


wilayah_uji_2024, hasil_2024, alokasi_2024, kasus_2024 = pengujian(
    2023, 2024, selisih_tc_2023, beban_2023, estimasi_insiden_2024,
)
wilayah_uji_2025, hasil_2025, alokasi_2025, kasus_2025 = pengujian(
    2024, 2025, selisih_tc_2024, beban_2024, wilayah_skrining["insiden_evaluasi"],
)

for tahun, wilayah, hasil in ((2024, wilayah_uji_2024, hasil_2024), (2025, wilayah_uji_2025, hasil_2025)):
    print(f"\nPengujian {tahun} pada {len(wilayah)} wilayah")
    print(hasil.sort_values("ekspektasi_kasus", ascending=False).round(2).to_string())

ringkasan = pd.DataFrame({
    "pengujian_2024": hasil_2024["ekspektasi_kasus"],
    "pengujian_2025": hasil_2025["ekspektasi_kasus"],
}).sort_values("pengujian_2025", ascending=False)
print("\nRingkasan ekspektasi kasus per 100.000 skrining")
print(ringkasan.round(1).to_string())

# Ketahanan pengujian 2024 terhadap pembagian out-of-fold.
hasil_per_ulangan = []
for prediksi in prediksi_oof_ulangan:
    _, hasil, _, _ = pengujian(2023, 2024, prediksi - y_2023, beban_2023, estimasi_insiden_2024)
    hasil_per_ulangan.append(hasil["ekspektasi_kasus"])
hasil_per_ulangan = pd.concat(hasil_per_ulangan, axis=1)
menang = (hasil_per_ulangan.loc["TRACK-TB"] >= hasil_per_ulangan.max()).sum()
print(f"\nTRACK-TB 2024 per pembagian: {hasil_per_ulangan.loc['TRACK-TB'].min():.1f}–{hasil_per_ulangan.loc['TRACK-TB'].max():.1f} kasus; "
      f"tertinggi pada {menang} dari {JUMLAH_ULANGAN_OOF} pembagian.")

for tahun, wilayah, alokasi in ((2024, wilayah_uji_2024, alokasi_2024), (2025, wilayah_uji_2025, alokasi_2025)):
    teratas = alokasi["TRACK-TB"].nlargest(6)
    print(f"\nPengujian {tahun}: {int(alokasi['TRACK-TB'].gt(0).sum())} wilayah menerima skrining TRACK-TB; "
          f"lima teratas menyerap {teratas.head(5).sum() / ANGGARAN_SKRINING:.0%}.")
    print(pd.DataFrame({
        "kabupaten_kota": frames[tahun].loc[teratas.index, "kabupaten_kota"],
        "skrining_TRACK-TB": teratas,
    }).to_string())

audit_tersedia = audit_tc_2025.dropna(subset=["selisih_tc_pp"]).sort_values(
    "selisih_tc_pp", ascending=False, kind="stable",
)
print(f"\nAudit 2025 memuat {len(audit_tersedia)} selisih TC. "
      f"{audit_tc_2025['selisih_tc_pp'].isna().sum()} wilayah belum punya selisih.")
print(audit_tersedia.head(8).round(2).to_string())



Pengujian 2024 pada 71 wilayah
                                      skrining  ekspektasi_kasus  kasus_per_1000_skrining  selisih_vs_merata  selisih_vs_penduduk
TRACK-TB                                100000            161.47                     1.61              32.01                44.91
Kepadatan kasus hilang (tanpa model)    100000            150.43                     1.50              20.96                33.87
TC rendah (tanpa model)                 100000            147.08                     1.47              17.61                30.52
Beban kasus hilang                      100000            129.80                     1.30               0.33                13.24
Merata                                  100000            129.47                     1.29               0.00                12.91
Berdasarkan penduduk                    100000            116.56                     1.17             -12.91                 0.00

Pengujian 2025 pada 57 wilayah
                          


TRACK-TB 2024 per pembagian: 154.9–172.2 kasus; tertinggi pada 10 dari 10 pembagian.

Pengujian 2024: 42 wilayah menerima skrining TRACK-TB; lima teratas menyerap 25%.
            kabupaten_kota  skrining_TRACK-TB
kode_bps                                     
1501               Kerinci               5549
1376       Kota Payakumbuh               5338
1373      Kota Sawah Lunto               4830
1303                 Solok               4801
1405                  Siak               4770
1804         Lampung Timur               4577

Pengujian 2025: 20 wilayah menerima skrining TRACK-TB; lima teratas menyerap 61%.
              kabupaten_kota  skrining_TRACK-TB
kode_bps                                       
1377           Kota Pariaman              24543
1301      Kepulauan Mentawai              11127
1373        Kota Sawah Lunto              10653
1376         Kota Payakumbuh               7879
1307                    Agam               6507
1809               Pesawaran               4

In [13]:
warna = {"Pengujian 2024": "#84b9f3", "Pengujian 2025": "#1757ac"}
urutan = ringkasan.index.tolist()
grafik_simulasi = go.Figure()
for nama, kolom, wilayah in (("Pengujian 2024", "pengujian_2024", wilayah_uji_2024), ("Pengujian 2025", "pengujian_2025", wilayah_uji_2025)):
    grafik_simulasi.add_trace(go.Bar(
        name=f"{nama} ({len(wilayah)} wilayah)",
        x=urutan,
        y=ringkasan.loc[urutan, kolom].tolist(),
        text=[f"{angka:.1f}" for angka in ringkasan.loc[urutan, kolom]],
        textposition="outside",
        marker_color=warna[nama],
        cliponaxis=False,
    ))
grafik_simulasi.update_yaxes(title_text="Ekspektasi kasus tambahan", range=[0, 1.2 * ringkasan.to_numpy().max()])
grafik_simulasi.update_xaxes(tickangle=-25, automargin=True)
grafik_simulasi.update_layout(
    title=f"Alokasi {ANGGARAN_SKRINING:,} skrining per strategi pada dua pengujian bergulir",
    barmode="group", template="plotly_white", height=430, margin=dict(l=55, r=20, t=90, b=90),
    legend=dict(orientation="h", y=1.08),
)
display({
    "application/vnd.plotly.v1+json": json.loads(grafik_simulasi.to_json()),
    "text/plain": "Grafik ekspektasi kasus per strategi pada pengujian 2024 dan 2025.",
}, raw=True)


Grafik ekspektasi kasus per strategi pada pengujian 2024 dan 2025.